In [1]:
import pandas as pd

In [2]:
shipments = pd.DataFrame({
    "shipment_id": [
        "S101", "S102", "S103", "S104", "S105",
        "S106", "S107", "S108", "S109", "S110",
        "S110"
    ],
    "warehouse": [
        "Pune", "Pune", "Mumbai", "Mumbai", "Pune",
        "Delhi", "Delhi", "Mumbai", "Pune", "Delhi",
        "Delhi"
    ],
    "region": [
        "West", "West", "West", "West", "West",
        "North", "North", "West", "West", "North",
        "North"
    ],
    "customer_id": [
        "C01", "C02", "C03", "C01", "C04",
        "C05", "C05", "C06", "C01", "C07",
        "C07"
    ],
    "status": [
        "Delivered", "Delivered", "Delayed", "Delivered", "Cancelled",
        "Delivered", "Delayed", "Delivered", "Delivered", "Cancelled",
        "Cancelled"
    ],
    "revenue": [
        2500, 1800, 3200, 2100, 1500,
        4000, 2800, 3500, 2200, 1900,
        1900
    ],
    "delivery_days": [
        2, 3, 6, 2, 1,
        4, 7, 3, 2, 1,
        1
    ]
})

### Inspection

In [3]:
shipments.head()

,shipment_id,warehouse,region,customer_id,status,revenue,delivery_days
0,S101,Pune,West,C01,Delivered,2500,2
1,S102,Pune,West,C02,Delivered,1800,3
2,S103,Mumbai,West,C03,Delayed,3200,6
3,S104,Mumbai,West,C01,Delivered,2100,2
4,S105,Pune,West,C04,Cancelled,1500,1


In [4]:
shipments.info()

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   shipment_id    11 non-null     str  
 1   warehouse      11 non-null     str  
 2   region         11 non-null     str  
 3   customer_id    11 non-null     str  
 4   status         11 non-null     str  
 5   revenue        11 non-null     int64
 6   delivery_days  11 non-null     int64
dtypes: int64(2), str(5)
memory usage: 748.0 bytes


In [5]:
shipments[shipments.duplicated()]

,shipment_id,warehouse,region,customer_id,status,revenue,delivery_days
10,S110,Delhi,North,C07,Cancelled,1900,1


In [6]:
shipments[shipments['shipment_id'] == 'S110']

,shipment_id,warehouse,region,customer_id,status,revenue,delivery_days
9,S110,Delhi,North,C07,Cancelled,1900,1
10,S110,Delhi,North,C07,Cancelled,1900,1


In [7]:
shipments.drop_duplicates(inplace=True)

In [8]:
shipments.count()

shipment_id      10
warehouse        10
region           10
customer_id      10
status           10
revenue          10
delivery_days    10
dtype: int64

### KPI

Filter rows where status is not cancelled

In [9]:
filtered_shipments = shipments[shipments['status'] != 'Cancelled']

Then create a warehouse-level KPI table where one row = one warehouse containing:

1. total_shipments
2. total_revenue
3. avg_revenue
4. avg_delivery_days
5. max_delivery_days
6. unique_customers

In [17]:
shipments.groupby('warehouse', as_index=False).agg(
    total_shipments=('shipment_id', 'count'),
)

,warehouse,total_shipments
0,Delhi,3
1,Mumbai,3
2,Pune,4


In [16]:
grouped = filtered_shipments.groupby('warehouse', as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    max_delivery_days=('delivery_days', 'max'),
    unique_customers=('customer_id', 'nunique')
)

In [11]:
grouped

,warehouse,total_shipments,total_revenue,avg_revenue,avg_delivery_days,max_delivery_days,unique_customers
0,Delhi,2,6800,3400.000000,5.500000,7,1
1,Mumbai,3,8800,2933.333333,3.666667,6,3
2,Pune,3,6500,2166.666667,2.333333,3,2


Which warehouse has the highest total revenue?

In [12]:
grouped.nlargest(n=1, columns=['total_revenue'])

,warehouse,total_shipments,total_revenue,avg_revenue,avg_delivery_days,max_delivery_days,unique_customers
1,Mumbai,3,8800,2933.333333,3.666667,6,3


Which warehouse has the highest average delivery time?

In [13]:
grouped.nlargest(n=1, columns=['avg_delivery_days'])

,warehouse,total_shipments,total_revenue,avg_revenue,avg_delivery_days,max_delivery_days,unique_customers
0,Delhi,2,6800,3400.0,5.5,7,1


Show only warehouses with total revenue > ₹7,000

In [14]:
grouped[grouped['total_revenue'] > 7000]

,warehouse,total_shipments,total_revenue,avg_revenue,avg_delivery_days,max_delivery_days,unique_customers
1,Mumbai,3,8800,2933.333333,3.666667,6,3


Create a second KPI table at region level containing total revenue, average delivery days, and unique customers.

In [15]:
filtered_shipments.groupby('region', as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    avg_delivery_days=('delivery_days', 'mean'),
    unique_customers=('customer_id', 'nunique')
)

,region,total_revenue,avg_delivery_days,unique_customers
0,North,6800,5.5,1
1,West,15300,3.0,4


What happens if I don't remove the duplicate S110? \
-> Every KPI would get added by duplicate values \
Should a cancelled shipment count toward total_shipments even though it doesn't contribute revenue? \
-> No as shipment is cancelled it should not count to total_shipments and revenue \
What exactly does one row of my final warehouse table represent? \
-> One row = One warehouse